#### READ RAW `ORDER_STATUS` FROM VOLUME

#### DEFINE SCHEMA FOR `ORDER_STATUS`

In [0]:
orders_schema = "Order_ID INT, Order_Date STRING, Status STRING, Total_Amount STRING"

In [0]:
orders_df = (spark.read \
    .option("header", "true")
    .schema(orders_schema)
    .csv('/Volumes/motor/landing/orders/')
)
# Show first 10 records
display(orders_df)

In [0]:
from pyspark.sql.functions import to_date, regexp_replace, col, current_timestamp

orders_trans_df = (
    orders_df.withColumnRenamed("Order_ID", "OrderID")
    .withColumn("TotalAmount", regexp_replace(col("Total_Amount"), ",", "").cast("float"))
    .withColumn("OrderDate", to_date("Order_Date", "M/d/yyyy"))
    .withColumn('load_timestamp', current_timestamp())
    .drop("Order_Date","Total_Amount")
)
display(orders_trans_df)

In [0]:
orders_trans_df.write.mode('append').saveAsTable('MOTOR.SILVER.ORDER_STATUS_RAW_SCD_1')

In [0]:
%sql
ALTER TABLE MOTOR.SILVER.ORDER_STATUS_RAW_SCD_1
SET TBLPROPERTIES (delta.enableChangeDataFeed = true);

In [0]:
%python
dbutils.notebook.exit('EXITED SUCCESSFULLY')